In [ ]:
# Importación de las librerías requeridas
import pandas as pd
import numpy as np

c:\Users\USUARIO\OneDrive\Desktop\Cursos UNIR\Curso python


In [ ]:
# Carga de datos utilizando las rutas absolutas exigidas
df_sales = pd.read_csv('/workspace/sales.csv')
df_inventories = pd.read_csv('/workspace/inventories.csv')
df_satisfaction = pd.read_csv('/workspace/satisfaction.csv')

# Limpieza
df_sales = df_sales.dropna()
df_inventories = df_inventories.dropna()
df_satisfaction = df_satisfaction.dropna()

print("Datos cargados y limpiados exitosamente.")

# Validación explícita de estructura (Sugerencia Criterio 1)
print("\n--- Estructura de los DataFrames ---")
print(f"Ventas: {df_sales.shape[0]} filas, {df_sales.shape[1]} columnas. Columnas: {list(df_sales.columns)}")
print(f"Inventarios: {df_inventories.shape[0]} filas, {df_inventories.shape[1]} columnas. Columnas: {list(df_inventories.columns)}")
print(f"Satisfacción: {df_satisfaction.shape[0]} filas, {df_satisfaction.shape[1]} columnas. Columnas: {list(df_satisfaction.columns)}")

Datos cargados y limpiados exitosamente.
Index(['ID_Tienda', 'Producto', 'Cantidad_Vendida', 'Precio_Unitario',
       'Fecha_Venta'],
      dtype='str')


In [ ]:
# 1. Calcular ingresos totales por transacción
df_sales['Ingresos'] = df_sales['Cantidad_Vendida'] * df_sales['Precio_Unitario']

# 2. Ventas totales (unidades) por producto y por tienda
ventas_prod_tienda = df_sales.groupby(['ID_Tienda', 'Producto'])['Cantidad_Vendida'].sum().reset_index()
print("--- Ventas Totales por Producto y Tienda ---")
display(ventas_prod_tienda)

# 3. Ingresos totales agrupados por tienda
ingresos_por_tienda = df_sales.groupby('ID_Tienda')['Ingresos'].sum().reset_index()
print("\n--- Ingresos Totales por Tienda ---")
display(ingresos_por_tienda)

# 4. Resumen estadístico general de las ventas
resumen_estadistico = df_sales.describe()
print("\n--- Resumen Estadístico de Ventas ---")
display(resumen_estadistico)

--- Resumen Estadístico de Ventas ---


,ID_Tienda,Cantidad_Vendida,Precio_Unitario,Ingresos
count,10.000000,10.000000,10.000000,10.000000
mean,3.000000,25.000000,190.000000,5050.000000
std,1.490712,9.128709,87.559504,3361.960407
min,1.000000,10.000000,100.000000,1000.000000
25%,2.000000,20.000000,100.000000,2625.000000
50%,3.000000,25.000000,200.000000,3500.000000
75%,4.000000,30.000000,275.000000,7875.000000
max,5.000000,40.000000,300.000000,10500.000000


In [ ]:
# Unir las ventas totales con el dataframe de inventarios
df_inventories = df_inventories.merge(ventas_prod_tienda, on=['ID_Tienda', 'Producto'], how='left')
df_inventories['Cantidad_Vendida'] = df_inventories['Cantidad_Vendida'].fillna(0)

# Calcular rotación por producto
df_inventories['Rotacion'] = df_inventories['Cantidad_Vendida'] / df_inventories['Stock_Disponible']

# Filtrar tiendas con niveles críticos
inventario_critico = df_inventories[df_inventories['Rotacion'] < 0.10]
tiendas_criticas = inventario_critico['ID_Tienda'].unique()

print("--- Tiendas con Niveles Críticos de Inventario (<10% de rotación) ---")
print(tiendas_criticas if len(tiendas_criticas) > 0 else "Ninguna tienda presenta niveles críticos.")

# Rotación agregada por tienda (Corrección Criterio 2)
rotacion_agregada = df_inventories.groupby('ID_Tienda')[['Cantidad_Vendida', 'Stock_Disponible']].sum().reset_index()
rotacion_agregada['Rotacion_Total'] = rotacion_agregada['Cantidad_Vendida'] / rotacion_agregada['Stock_Disponible']

print("\n--- Resumen de Rotación Agregada por Tienda ---")
display(rotacion_agregada[['ID_Tienda', 'Rotacion_Total']])

--- Tiendas con Niveles Críticos de Inventario (<10% de rotación) ---
Ninguna tienda presenta niveles críticos.


In [20]:
# Relacionar el rendimiento de ventas (ingresos) con la satisfacción usando ID_Tienda
rendimiento_tiendas = ingresos_por_tienda.merge(df_satisfaction, on='ID_Tienda')

# Filtrar tiendas con baja satisfacción (< 60%)
tiendas_baja_satisfaccion = rendimiento_tiendas[rendimiento_tiendas['Satisfacción_Promedio'] < 60]

print("--- Tiendas con Satisfacción Menor al 60% ---")
display(tiendas_baja_satisfaccion)

print("\n--- Recomendaciones Estratégicas ---")
print("1. Las tiendas identificadas requieren auditorías urgentes de servicio al cliente.")
print("2. Se sugiere revisar los tiempos de espera y la capacitación del personal.")
print("3. Cruzar estos datos con el nivel de rotación de inventario para descartar desabastecimiento como causa de insatisfacción.")

--- Tiendas con Satisfacción Menor al 60% ---


,ID_Tienda,Ingresos,Satisfacción_Promedio,Fecha_Evaluación
4,5,13000,55,2023-01-15



--- Recomendaciones Estratégicas ---
1. Las tiendas identificadas requieren auditorías urgentes de servicio al cliente.
2. Se sugiere revisar los tiempos de espera y la capacitación del personal.
3. Cruzar estos datos con el nivel de rotación de inventario para descartar desabastecimiento como causa de insatisfacción.


In [ ]:
# Extraer la columna de ingresos como un array de Numpy usando .to_numpy()
ingresos_array = ingresos_por_tienda['Ingresos'].to_numpy()

# 1. Cálculos estadísticos con Numpy (Aclaración Criterio 3)
mediana_ventas = np.median(ingresos_array)
desviacion_ventas = np.std(ingresos_array)

print(f"Mediana de los ingresos totales por tienda (Numpy): ${mediana_ventas:.2f}")
print(f"Desviación estándar de los ingresos totales por tienda (Numpy): ${desviacion_ventas:.2f}")

# 2. Simulación de proyecciones futuras
np.random.seed(42) 
simulacion_crecimiento = np.random.uniform(0.95, 1.15, size=len(ingresos_array))
proyeccion_ventas = ingresos_array * simulacion_crecimiento

print("\n--- Proyección de Ventas Futuras Simuladas ---")
for tienda, proyeccion in zip(ingresos_por_tienda['ID_Tienda'], proyeccion_ventas):
    print(f"Tienda ID {tienda} | Proyección: ${proyeccion:.2f}")

# 3. Estadísticas básicas sobre las proyecciones (Corrección Criterio 4)
media_proy = np.mean(proyeccion_ventas)
std_proy = np.std(proyeccion_ventas)

print("\n--- Estadísticas de las Proyecciones Futuras ---")
print(f"Media de las proyecciones: ${media_proy:.2f}")
print(f"Desviación estándar de las proyecciones: ${std_proy:.2f}")

Mediana de ventas totales (Numpy): $10500.00
Desviación estándar de ventas (Numpy): $2973.21

--- Proyección de Ventas Futuras Simuladas ---
Tienda ID 1 | Proyección: $5124.54
Tienda ID 2 | Proyección: $11971.50
Tienda ID 3 | Proyección: $9867.59
Tienda ID 4 | Proyección: $13906.51
Tienda ID 5 | Proyección: $12755.65
